# ETL com Python - Dados de Leads da Internacional
Este notebook realiza o processo de ETL sobre os dados exportados do CRM da corretora Internacional.

In [6]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from Utils.io import save_csv

# Caminho base do projeto
BASE_DIR = Path.cwd().parent
raw_dir = BASE_DIR / "Data" / "RAW"
processed_dir = BASE_DIR / "Data" / "PROCESSED"
final_dir = BASE_DIR / "Data" / "FINAL"


## 1. Leitura do CSV exportado do CRM

In [2]:
# Leitura
file_path = raw_dir / 'Leads_Simulados.csv'
df = pd.read_csv(file_path, parse_dates=['data_cadastro'])
df.head()

,lead_id,origem,data_cadastro,status_conversao,dias_ate_1o_trade,valor_deposito,perfil,pais
0,2001,Orgânico,2025-03-29,Não Convertido,NaN,0.00,NaN,Brasil
1,2002,Instagram,2025-03-10,Não Convertido,NaN,0.00,NaN,Portugal
2,2003,Orgânico,2025-02-20,Convertido,6.0,7985.78,Scalper,Espanha
3,2004,Facebook,2025-03-13,Convertido,7.0,6026.72,Investidor,Brasil
4,2005,Orgânico,2025-01-28,Convertido,6.0,4513.74,Investidor,Espanha


## 2. Tratamento e padronização de dados

In [3]:
df['dias_ate_1o_trade'] = pd.to_numeric(df['dias_ate_1o_trade'], errors='coerce')
df['valor_deposito'] = pd.to_numeric(df['valor_deposito'], errors='coerce')
df['foi_convertido'] = df['status_conversao'] == 'Convertido'
df['ano_mes_cadastro'] = df['data_cadastro'].dt.to_period('M').astype(str)
df.head()

,lead_id,origem,data_cadastro,status_conversao,dias_ate_1o_trade,valor_deposito,perfil,pais,foi_convertido,ano_mes_cadastro
0,2001,Orgânico,2025-03-29,Não Convertido,NaN,0.00,NaN,Brasil,False,2025-03
1,2002,Instagram,2025-03-10,Não Convertido,NaN,0.00,NaN,Portugal,False,2025-03
2,2003,Orgânico,2025-02-20,Convertido,6.0,7985.78,Scalper,Espanha,True,2025-02
3,2004,Facebook,2025-03-13,Convertido,7.0,6026.72,Investidor,Brasil,True,2025-03
4,2005,Orgânico,2025-01-28,Convertido,6.0,4513.74,Investidor,Espanha,True,2025-01


## 3. Geração de indicadores por canal

In [4]:
agg_canais = df.groupby('origem').agg({
    'lead_id': 'count',
    'foi_convertido': 'mean',
    'valor_deposito': 'mean'
}).rename(columns={
    'lead_id': 'total_leads',
    'foi_convertido': 'taxa_conversao',
    'valor_deposito': 'deposito_medio'
}).reset_index()

agg_canais

,origem,total_leads,taxa_conversao,deposito_medio
0,Blog/SEO,4,1.000000,4294.650000
1,E-mail Marketing,14,0.500000,2080.732143
2,Facebook,44,0.704545,4069.480909
3,Google Ads,37,0.594595,2742.562973
4,Indicação,6,0.666667,2263.598333
5,Instagram,42,0.690476,3692.626667
6,LinkedIn,21,0.714286,2957.020000
7,Orgânico,43,0.744186,3788.528140
8,TikTok,34,0.529412,2616.296471
9,Twitter,13,0.384615,1784.475385


## 4. Exportação dos dados tratados para uso no Power BI ou Streamlit

In [5]:

# Salva os arquivos 
df.to_csv(processed_dir / "leads_processados_para_pbi.csv", index=False)
agg_canais.to_csv(final_dir / "indicadores_por_canal.csv", index=False)